# 🌿 PlantGuard — Colab Training Notebook
CDS6334 VIP · Track A · Group 17 (TT2L)

**Runtime → Change runtime type → GPU**, then **Runtime → Run all**.

This notebook clones the repo, downloads PlantVillage, trains the MobileNetV2 transfer model (+ baseline), evaluates it, and lets you download the trained model and result figures.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Get the code
Either clone your repo, **or** upload the `plantguard/` folder to Colab and `cd` into it. Replace the URL below.

In [ ]:
# Option A: clone your repository
# !git clone https://github.com/<your-username>/plantguard.git
# %cd plantguard

# Option B: if you uploaded the folder manually, just set the path:
%cd /content/plantguard
!ls

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt kagglehub

## 4. Download PlantVillage dataset
Uses kagglehub. You may be prompted for Kaggle credentials the first time (Account → Create New API Token → upload `kaggle.json`).

In [ ]:
!python -m src.download_data --source kaggle

## 5. Build the stratified 70/15/15 split

In [ ]:
!python -m src.data_prep --build-split

## 6. Train
Two-phase transfer learning for MobileNetV2, plus the from-scratch baseline CNN for the report comparison. Full training on the complete dataset typically takes ~30–60 min on a Colab T4.

In [ ]:
!python -m src.train --baseline

## 7. Evaluate
Generates metrics, the confusion matrix, Grad-CAM panels, the misclassified gallery, and the baseline comparison.

In [ ]:
!python -m src.evaluate

## 8. Preview the results inline

In [ ]:
import json, glob
from IPython.display import Image, display
print(open('outputs/metrics.json').read())
for f in ['outputs/confusion_matrix.png','outputs/gradcam_examples.png','outputs/misclassified_gallery.png']:
    try: display(Image(f))
    except Exception as e: print('missing', f, e)

## 9. Download trained model + outputs

In [ ]:
import shutil
shutil.make_archive('plantguard_artifacts','zip','.','models')
shutil.make_archive('plantguard_outputs','zip','.','outputs')
from google.colab import files
files.download('plantguard_artifacts.zip')
files.download('plantguard_outputs.zip')

## 10. (Optional) Launch the Streamlit demo via a tunnel
Colab can't serve Streamlit directly; use a tunnel such as localtunnel.

In [ ]:
# !npm install -g localtunnel
# !streamlit run app/streamlit_app.py &>/content/log.txt &
# !npx localtunnel --port 8501